## **條件、迴圈與串列應用** (50 min)
*   中斷機制
*   for 迴圈與串列
*   串列綜合表達式
*   二維串列







---


### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("APCS_7-1a")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / APCS_7-1a.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

### **APCS 7-1a 中斷機制**
#### **題目說明：**
#### 從1~100中, 找到第一個是10的倍數並印出

#### **解題想法**
*   break 語法



In [ ]:
#start-APCS_7-1a


In [ ]:
make_submit_button("APCS_7-1a")


<br>

### **APCS 7-1b 中斷機制**
#### **題目說明：**
#### 從1~100中, 找出所有奇數並印出

#### **解題想法**
*   continu 語法


In [ ]:
#start-APCS_7-1b


In [ ]:
make_submit_button("APCS_7-1b")


<br>

### **APCS 7-2 for 迴圈與串列**

#### **搭配可迭代物件**
* 串列
```python
for i in [1, 2, 3, 4, 5]:
  print(i)
```
* 字串
```python
for i in "12345":
  print(i)
```
* range物件
```python
for i in range(5):
  print(i)
```
* 連續IPO
```python
for i in sys.stdin:
  print(i)
```

<br>

#### **題目說明：**
#### 輸入一串英文的句子，請找出句子中共有幾個母音(a, e, i , o, u)
#### 例如："Hello, world"，一共有 e, o, o 三個母音，因此輸出結果為 3

``` python
"Hello, word"
Total: 3
```

#### **解題想法**
*   此題不需要使用連續 IPO
*   輸入的測試句子需要檢查母音，因此可以創造一個母音的串列
*   在 for 迴圈直接讀入測試句子的每個文字在實作上會比較簡單



In [ ]:
#start


#### **判別元素是否有在串列裡**
```python
元素 in 串列
```

In [ ]:
#start-APCS_7-2


In [ ]:
make_submit_button("APCS_7-2")


<br>

### **APCS 7-3 串列綜合表達式**

#### **基本格式**
```python
[expression for item in iterable]
```
*   expression 可以是變數或運算式，代表每一次迴圈要做的事情
*   item 則是從可迭代物件（iterable）中取出的元素
*   iterable 的內容如同 range(), list 或字串，代表一個可迭代的物件、資料集合

#### **優點**
*   將 for 迴圈包裝在一行串列當中，縮減程式碼的長度
*   具有較高的程式執行效率

<br>

#### **什麼時候使用?**
*   初始化數字串列
*   與串列相關的計算，且計算過程重複性質高
*   需要縮短程式碼，把握時間寫更多程式

<br>

#### **題目說明：**
#### 輸入一個 N 值，產生範圍 1~3*N 之間所有 3 的倍數，並存放於串列

#### **解題想法**
*   綜合表達式中的 expression 除了單一變數外，也可以進行運算，並且會將運算的結果，轉存為串列中的每一個元素。



*   輸入一個 N 值
*   會輸出 N, 2*N, 3*N … 的串列資料



#### **標準寫法**



In [ ]:
#start


#### **串列綜合表達式寫法**

In [ ]:
#start-APCS_7-3


In [ ]:
make_submit_button("APCS_7-3")


<br>

### **APCS 7-4 二維串列**
| |y 索引值增加的方向 |
|-|-|
| | [ |
| | [90,   95,   75,   80,  85] #John |
| x 索引值增加的方向 | [60,   75,   99,   55,  90] #Mary |
| | [80,   80,   30,   95,  90]] #Oliver  |
| | ] |

```python
[ #[國文, 英文, 數學, 自然, 社會]
   [90,   95,   75,   80,  85], #John
   [60,   75,   99,   55,  90], #Mary
   [80,   80,   30,   95,  90], #Oliver
]
```

<br>

#### **題目說明：**
#### 使用二維串列, 回答下列問題:
*   請問 John 的數學成績
*   請問 Mary 的自然成績
*   請問 Oliver 的國語成績
*   score[0][3]+score[1][3]+score[2][3] 請問這是在計算什麼?
*   計算三位學生成績的平均分數


#### **解題想法**
*   使用二維串列建立三個人的成績
*   計算平均分數時需要加總，可以使用 sum() 計算，也可以使用 for 迴圈累加五科分數




In [ ]:
#start-APCS_7-4


In [ ]:
make_submit_button("APCS_7-4")


<br>

## **重點複習**
*   程式的流程控制，可以使用 break 和 continue 的語法指令，中斷迴圈原本的執行順序
*   for 迴圈可以使用 range()、串列和字串的方式讀入元素
*   串列綜合表達式可以縮減程式碼的長度，常用在建立數列串列或是具有重複性值高的串列運算
*   二維串列可以儲存更多種類型的資料，例如表格化的資料、模擬地圖移動、圖形計算等

